# Data postprocessing: updated fingerprints and reference coordinates

This notebook loads the **final reference fingerprint state after a completed update batch** and aligns its updated RSSI vectors with the immutable coordinates from the labeled bootstrap set. It reads the post-transition `current_prepared.json` or `reference_prepared.json` artifact through GUFU's shared loader, so AP-column migration and per-cell value provenance stay attached to the data.

Run from the repository root after `gufu.py` has completed at least one update batch. This notebook inspects data only: it does not fit a localization model or write pipeline artifacts.


## Step 1 — Select a completed batch snapshot

Copy the exact snapshot directory printed by `gufu.py` cell 8 into `SNAPSHOT_DIRECTORY` below. It should contain either `manifest.json` (the `lp_streaming` pipeline) or `snapshot_manifest.json` (the `full`/`stepwise` orchestrator). The notebook rejects bootstrap-only or incomplete snapshots.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from gufu_artifacts import canonical_json_digest, sha256_file
from gufu_data import load_prepared_dataset

# Set this to the completed batch snapshot directory printed by gufu.py cell 8.
# SNAPSHOT_DIRECTORY = "/home/oem/GUFU/output_fromGF_12/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-spatial-v2-softplus-cosine-t0p1-nw1-terminal-node-v1-d256-z32-drop0p1-data1be80e24f66a5214/streaming_pipeline/snapshots/0001-fingerprint1-70c080725fa5"
SNAPSHOT_DIRECTORY = "/home/oem/GUFU/output_fromGF_12/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-spatial-v2-softplus-cosine-t0p1-nw1-terminal-node-v1-d256-z32-drop0p1-data1be80e24f66a5214/streaming_pipeline/snapshots/0001-fingerprint1-3ac8b538d556"
# Example: Path("output/notebook_runs/GF/<run-id>/streaming_pipeline/snapshots/<snapshot-id>")

if SNAPSHOT_DIRECTORY is None:
    raise ValueError(
        "Set SNAPSHOT_DIRECTORY to the completed batch snapshot directory printed by gufu.py cell 8."
    )
SNAPSHOT_DIRECTORY = Path(SNAPSHOT_DIRECTORY)
if not SNAPSHOT_DIRECTORY.is_dir():
    raise FileNotFoundError(f"snapshot directory does not exist: {SNAPSHOT_DIRECTORY}")

print(f"Selected snapshot: {SNAPSHOT_DIRECTORY}")


Selected snapshot: /home/oem/GUFU/output_fromGF_12/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-spatial-v2-softplus-cosine-t0p1-nw1-terminal-node-v1-d256-z32-drop0p1-data1be80e24f66a5214/streaming_pipeline/snapshots/0001-fingerprint1-3ac8b538d556


## Step 2 — Verify the snapshot and load its updated reference dataset

The final prepared artifact is used here because it is already remapped to the post-update AP vocabulary. The intermediate `updated_fingerprints.json` is created before AP-vocabulary migration in the standard orchestrator, so it can still use the previous batch's vocabulary.


In [2]:
if (SNAPSHOT_DIRECTORY / "manifest.json").is_file():
    from streaming_pipeline import StreamingBatchSnapshotManifest

    snapshot_format = "lp_streaming"
    manifest_path = SNAPSHOT_DIRECTORY / "manifest.json"
    manifest = StreamingBatchSnapshotManifest.model_validate_json(
        manifest_path.read_text(encoding="utf-8")
    )
    marker = json.loads((SNAPSHOT_DIRECTORY / "COMPLETED.json").read_text(encoding="utf-8"))
    assert marker.get("snapshot_id") == manifest.snapshot_id
    assert marker.get("manifest_sha256") == sha256_file(manifest_path)
    for artifact in manifest.artifacts:
        artifact_path = SNAPSHOT_DIRECTORY / artifact.path
        assert artifact_path.is_file(), f"snapshot artifact is missing: {artifact.path}"
        assert artifact_path.stat().st_size == artifact.size_bytes
        assert sha256_file(artifact_path) == artifact.sha256
    updated_prepared_path = SNAPSHOT_DIRECTORY / manifest.current_state.reference_prepared
    provenance_path = SNAPSHOT_DIRECTORY / manifest.current_state.reference_value_provenance
    expected_dataset_id = manifest.dataset_id
    expected_batch_id = manifest.batch_id
    expected_vocabulary = manifest.vocabulary_after
    expected_scan_ids = manifest.reference_scan_ids
    vocabulary_before = manifest.vocabulary_before
    update_scan_count = len(manifest.update_scan_ids)
    snapshot_id = manifest.snapshot_id

elif (SNAPSHOT_DIRECTORY / "snapshot_manifest.json").is_file():
    from pred import CompletionMarker, PipelineSnapshotManifest

    snapshot_format = "full_or_stepwise"
    manifest_path = SNAPSHOT_DIRECTORY / "snapshot_manifest.json"
    manifest = PipelineSnapshotManifest.model_validate_json(
        manifest_path.read_text(encoding="utf-8")
    )
    marker = CompletionMarker.model_validate_json(
        (SNAPSHOT_DIRECTORY / "COMPLETED.json").read_text(encoding="utf-8")
    )
    assert marker.snapshot_id == manifest.snapshot_id
    assert marker.manifest_sha256 == sha256_file(manifest_path)
    for artifact in manifest.artifacts:
        artifact_path = SNAPSHOT_DIRECTORY / artifact.relative_path
        assert artifact_path.is_file(), f"snapshot artifact is missing: {artifact.relative_path}"
        assert sha256_file(artifact_path) == artifact.sha256
    if manifest.snapshot_kind != "batch":
        raise ValueError("Select a completed batch snapshot, not the bootstrap snapshot.")
    artifact_paths = dict(manifest.artifact_index)
    updated_prepared_path = SNAPSHOT_DIRECTORY / artifact_paths["current_prepared"]
    provenance_path = SNAPSHOT_DIRECTORY / artifact_paths["current_value_provenance"]
    expected_dataset_id = manifest.dataset_id
    expected_batch_id = manifest.batch_id
    expected_vocabulary = manifest.bssid_vocabulary
    expected_scan_ids = None  # The snapshot manifest records the count; the prepared artifact supplies the ordered IDs.
    transition_path = SNAPSHOT_DIRECTORY / "transition" / "transition_manifest.json"
    transition_record = json.loads(transition_path.read_text(encoding="utf-8"))
    vocabulary_before = tuple(transition_record["summary"]["vocabulary_before"])
    update_scan_count = manifest.update_scan_count
    snapshot_id = manifest.snapshot_id
else:
    raise ValueError(
        "Snapshot must contain manifest.json or snapshot_manifest.json and COMPLETED.json."
    )

updated_reference = load_prepared_dataset(
    updated_prepared_path,
    value_provenance_path=provenance_path,
)
scan_ids = tuple(scan.scan_id for scan in updated_reference.scans)
assert updated_reference.config.dataset_id == expected_dataset_id
assert updated_reference.config.batch_role == "bootstrap_reference"
assert updated_reference.bssid_vocabulary == tuple(expected_vocabulary)
assert len(scan_ids) == len(set(scan_ids))
assert all(scan.value_provenance is not None for scan in updated_reference.scans)
if expected_scan_ids is not None:
    assert scan_ids == tuple(expected_scan_ids)
else:
    assert len(scan_ids) == manifest.reference_scan_count
assert updated_reference.evaluation_positions == () or all(
    position.scan_id not in set(scan_ids) for position in updated_reference.evaluation_positions
)

print(f"Snapshot format: {snapshot_format}")
print(f"Snapshot ID: {snapshot_id}")
print(f"Updated batch: {expected_batch_id}")
print(f"Prepared artifact: {updated_prepared_path}")
print(f"Value provenance sidecar: {provenance_path}")
print(updated_reference.summary())


Snapshot format: lp_streaming
Snapshot ID: streambatch:9b9973850ba21d09e20e36e66056d5ff56fcc48160a0d09736bf3ac8b538d556
Updated batch: fingerprint1
Prepared artifact: /home/oem/GUFU/output_fromGF_12/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-spatial-v2-softplus-cosine-t0p1-nw1-terminal-node-v1-d256-z32-drop0p1-data1be80e24f66a5214/streaming_pipeline/snapshots/0001-fingerprint1-3ac8b538d556/reference_prepared.json
Value provenance sidecar: /home/oem/GUFU/output_fromGF_12/notebook_runs/GF/real-GF-fingerprint0-ae-v2-mse-rssi-pairs-v1-spatial-v2-softplus-cosine-t0p1-nw1-terminal-node-v1-d256-z32-drop0p1-data1be80e24f66a5214/streaming_pipeline/snapshots/0001-fingerprint1-3ac8b538d556/reference_value_provenance.json
{'dataset_id': 'GF', 'batch_id': 'fingerprint0', 'batch_role': 'bootstrap_reference', 'scan_count': 2078, 'ap_count': 1041, 'reference_label_count': 2078, 'evaluation_truth_count': 0, 'dropped_record_count': 0}


## Step 3 — Inspect what the completed batch changed

The reference scan IDs and original reference coordinates remain fixed. The AP vocabulary may change after AP admission or forgetting. Updated rows keep a per-cell provenance label so measured RSSI, decoder predictions, edge predictions, and missing values are distinguishable.


In [3]:
bssid_vocabulary = updated_reference.bssid_vocabulary
reference_position_ids = tuple(position.scan_id for position in updated_reference.reference_positions)

assert len(updated_reference.scans) == len(scan_ids)
assert len(bssid_vocabulary) == len(set(bssid_vocabulary))
assert set(reference_position_ids) <= set(scan_ids)
assert len(reference_position_ids) == len(set(reference_position_ids))
assert len(bssid_vocabulary) == len(expected_vocabulary)

added_bssids = tuple(bssid for bssid in bssid_vocabulary if bssid not in set(vocabulary_before))
removed_bssids = tuple(bssid for bssid in vocabulary_before if bssid not in set(bssid_vocabulary))

print(f"Reference scans retained: {len(scan_ids)}")
print(f"Original labeled reference coordinates: {len(reference_position_ids)}")
print(f"Incoming temporary scans consumed in batch: {update_scan_count}")
print(f"AP vocabulary: {len(vocabulary_before)} before -> {len(bssid_vocabulary)} after")
print(f"APs admitted: {len(added_bssids)}; APs forgotten: {len(removed_bssids)}")
print(f"Normalization: {updated_reference.config.normalization_mode}")
print(f"Rows still map to original scan IDs: {scan_ids[:3]}")


Reference scans retained: 2078
Original labeled reference coordinates: 2078
Incoming temporary scans consumed in batch: 1733
AP vocabulary: 1022 before -> 1041 after
APs admitted: 19; APs forgotten: 0
Normalization: paper_ae
Rows still map to original scan IDs: ('scan:dc7f29deee0a3ef0a656e7bf575d36157e772603f07f1737cc362d8ae532d3fe', 'scan:3d6e7516fb396a8f466059332dea349c7129ca3e2356a61a8355e6a8bc58373c', 'scan:62b353a32f5730f8a9720917fe5e179fdddb361e83851bb4a0cbc54556bff142')


## Step 4 — Align original coordinates by stable scan ID

Coordinates are joined to updated fingerprints by `scan_id`; repeated locations remain separate samples. A missing coordinate is displayed as `None`, not as `(0, 0)`. Only immutable `reference_positions` are treated as localization training labels.


In [4]:
reference_xy_by_scan_id = {
    position.scan_id: (position.x, position.y)
    for position in updated_reference.reference_positions
}

coordinate_rows = []
for row_index, scan in enumerate(updated_reference.scans):
    xy = reference_xy_by_scan_id.get(scan.scan_id)
    coordinate_rows.append(
        {
            "row": row_index,
            "scan_id": scan.scan_id,
            "x": None if xy is None else xy[0],
            "y": None if xy is None else xy[1],
            "label_available": xy is not None,
            "coordinate_role": scan.coordinate_role,
            "source_batch": scan.batch_id,
            "source_row": scan.source.source_row,
        }
    )

coordinate_table = pd.DataFrame(coordinate_rows).set_index("row")
assert tuple(coordinate_table["scan_id"]) == scan_ids
assert set(reference_xy_by_scan_id) == {
    scan.scan_id for scan in updated_reference.scans
    if scan.coordinate_role == "bootstrap_train"
}
print(f"Coordinate labels available: {len(reference_xy_by_scan_id)}/{len(scan_ids)}")
display(coordinate_table.head(12))


Coordinate labels available: 2078/2078


,scan_id,x,y,label_available,coordinate_role,source_batch,source_row
row,,,,,,,
0,scan:dc7f29deee0a3ef0a656e7bf575d36157e772603f...,458.651806,392.645723,True,bootstrap_train,fingerprint0,1
1,scan:3d6e7516fb396a8f466059332dea349c7129ca3e2...,458.731986,396.680500,True,bootstrap_train,fingerprint0,2
2,scan:62b353a32f5730f8a9720917fe5e179fdddb361e8...,460.125183,380.536226,True,bootstrap_train,fingerprint0,3
3,scan:9e6a20e0ecae343970251cdf768c0e5f8bd30ef45...,460.508937,384.570088,True,bootstrap_train,fingerprint0,4
4,scan:93f6c79a1172aa19f41501ee563732025b8ec4a74...,460.589117,388.604864,True,bootstrap_train,fingerprint0,5
5,scan:b842e5dabb08aeaeed3b1e1daf9879e3ed26177d4...,463.956830,392.459835,True,bootstrap_train,fingerprint0,6
6,scan:affa23d46be5a64ebb3bdfb4f43df81c793ad0453...,463.949153,396.494339,True,bootstrap_train,fingerprint0,7
7,scan:25f922e32b936ce294de1f4ea4f0ef371e0dcf7f5...,464.551167,380.354601,True,bootstrap_train,fingerprint0,8
8,scan:cdd49d3e2fff8a4fe87c0cfd33038f1898e05c284...,464.644681,384.388800,True,bootstrap_train,fingerprint0,9


## Step 5 — Inspect the post-update AP vocabulary

Each RSSI column is identified by its full canonical BSSID. The before/after sets below are derived from the completed transition, not from AP column positions.


In [5]:
column_for_bssid = {bssid: column for column, bssid in enumerate(bssid_vocabulary)}
assert len(column_for_bssid) == len(bssid_vocabulary)

vocabulary_changes = pd.DataFrame(
    [
        *({"change": "admitted", "bssid": bssid} for bssid in added_bssids),
        *({"change": "forgotten", "bssid": bssid} for bssid in removed_bssids),
    ],
    columns=["change", "bssid"],
)
print(f"Updated AP columns: {len(bssid_vocabulary)}")
print("First post-update vocabulary entries:")
for column, bssid in enumerate(bssid_vocabulary[:12]):
    print(f"column {column:3d}: {bssid}")
display(vocabulary_changes)


Updated AP columns: 1041
First post-update vocabulary entries:
column   0: 000B819A90BF
column   1: 000B819BBC05
column   2: 000B819EEBA5
column   3: 000B819FDD48
column   4: 000B81A08E12
column   5: 000B81A19DC9
column   6: 000B81A2359A
column   7: 000B81A3E6AC
column   8: 000B8575C2CF
column   9: 000B8575E238
column  10: 000B8575E23F
column  11: 000B8575E888


,change,bssid
0,admitted,0030441D5462
1,admitted,40EEDD2D9A6B
2,admitted,40EEDD2D9A6F
3,admitted,40EEDD2D9B6B
4,admitted,40EEDD2D9B6F
5,admitted,60F18AD612AB
6,admitted,60F18AD612AF
7,admitted,60F18AD61D6B
8,admitted,60F18AD61D6F
9,admitted,60F18AD659AB


## Step 6 — Assemble updated RSSI values and the measurement mask

`updated_rssi_dbm` contains the current dBm value for every reference scan/AP pair. `observation_mask` records whether the original scan physically measured that AP; it stays false for inferred values. Use provenance alongside the mask to distinguish inferred values from truly missing ones.


In [6]:
updated_rssi_dbm = torch.tensor(
    [scan.rss_dbm for scan in updated_reference.scans],
    dtype=torch.float32,
)
observation_mask = torch.tensor(
    [scan.observation_mask for scan in updated_reference.scans],
    dtype=torch.bool,
)
value_provenance = tuple(scan.value_provenance for scan in updated_reference.scans)

expected_shape = (len(scan_ids), len(bssid_vocabulary))
assert tuple(updated_rssi_dbm.shape) == expected_shape
assert tuple(observation_mask.shape) == expected_shape
assert all(len(row) == len(bssid_vocabulary) for row in value_provenance)
assert torch.isfinite(updated_rssi_dbm).all()

print(f"Updated RSSI matrix: {tuple(updated_rssi_dbm.shape)} dBm")
print(f"Measurement mask: {tuple(observation_mask.shape)}")
print(f"First updated RSSI row: {updated_rssi_dbm[0, :12].tolist()}")
print(f"First measurement-mask row: {observation_mask[0, :12].tolist()}")


Updated RSSI matrix: (2078, 1041) dBm
Measurement mask: (2078, 1041)
First updated RSSI row: [-119.90301513671875, -119.69670104980469, -118.64250183105469, -119.8772964477539, -119.74327850341797, -88.5, -116.96000671386719, -112.19944763183594, -119.8719711303711, -119.91309356689453, -120.19273376464844, -119.92835235595703]
First measurement-mask row: [False, False, False, False, False, True, False, False, False, False, False, False]


## Step 7 — Inspect one updated reference fingerprint

The table pairs each BSSID's updated value with its original measurement status and update origin. Decoder or edge predictions can fill an unmeasured cell without changing the fact that the source scan did not measure it.


In [7]:
example_row = 0
example_scan = updated_reference.scans[example_row]
example_fingerprint = pd.DataFrame(
    {
        "updated_rssi_dbm": example_scan.rss_dbm,
        "measured_in_original_scan": example_scan.observation_mask,
        "value_provenance": example_scan.value_provenance,
    },
    index=pd.Index(bssid_vocabulary, name="bssid"),
)
print(f"Scan ID: {example_scan.scan_id}")
print(f"Original coordinate: {reference_xy_by_scan_id.get(example_scan.scan_id)}")
display(example_fingerprint.head(20))


Scan ID: scan:dc7f29deee0a3ef0a656e7bf575d36157e772603f07f1737cc362d8ae532d3fe
Original coordinate: (458.65180591022914, 392.64572300816883)


,updated_rssi_dbm,measured_in_original_scan,value_provenance
bssid,,,
000B819A90BF,-119.903015,False,decoder_prediction
000B819BBC05,-119.696701,False,decoder_prediction
000B819EEBA5,-118.642502,False,decoder_prediction
000B819FDD48,-119.877296,False,decoder_prediction
000B81A08E12,-119.743279,False,decoder_prediction
000B81A19DC9,-88.500000,True,measured
000B81A2359A,-116.960007,False,decoder_prediction
000B81A3E6AC,-112.199448,False,decoder_prediction
000B8575C2CF,-119.871971,False,decoder_prediction


## Step 8 — Count measured, inferred, and missing cells

These counts describe cell provenance, not scan count. The physical observation mask remains a separate record of which values came from measurements.


In [8]:
from collections import Counter

provenance_counts = Counter(
    origin
    for row in value_provenance
    for origin in row
)
provenance_table = pd.DataFrame(
    [{"value_provenance": name, "cell_count": count}
     for name, count in sorted(provenance_counts.items())]
)
assert set(provenance_counts) <= {
    "measured", "decoder_prediction", "edge_prediction", "missing"
}
print(f"Physically measured cells: {int(observation_mask.sum())}")
print(f"Total fingerprint cells: {updated_rssi_dbm.numel()}")
display(provenance_table)


Physically measured cells: 82767
Total fingerprint cells: 2163198


,value_provenance,cell_count
0,decoder_prediction,2080431
1,measured,82767


## Step 9 — Produce the named AE input representation

`ae_matrix()` applies the dataset's recorded normalization and uses provenance to include decoder/edge estimates as input values. The physical observation mask is not used to guess missingness from RSSI numbers.


In [9]:
ae_input = torch.tensor(updated_reference.ae_matrix(), dtype=torch.float32)
assert tuple(ae_input.shape) == expected_shape
assert torch.isfinite(ae_input).all()

print(f"Normalization mode: {updated_reference.config.normalization_mode}")
print(f"AE input shape: {tuple(ae_input.shape)}")
print(f"AE input range: {ae_input.min().item():.6f} to {ae_input.max().item():.6f}")
print(f"First normalized row: {ae_input[0, :12].tolist()}")


Normalization mode: paper_ae
AE input shape: (2078, 1041)
AE input range: -0.051865 to 0.808333
First normalized row: [0.0008082072017714381, 0.0025274911895394325, 0.011312484741210938, 0.0010225295554846525, 0.0021393457427620888, 0.26249998807907104, 0.025333277881145477, 0.06500460207462311, 0.001066907192580402, 0.0007242202991619706, -0.00160611467435956, 0.00059706368483603]


## Step 10 — Build a coordinate-aligned localization table

`aligned_reference_table` contains one row per retained original reference scan, its original `(x, y)` when available, and all updated RSSI columns. `localization_training_table` selects only rows with bootstrap labels. Keep `scan_id`, the ordered BSSID vocabulary, and normalization metadata with any exported table.


In [10]:
coordinate_frame = pd.DataFrame(
    [
        {
            "scan_id": scan_id,
            "x": reference_xy_by_scan_id.get(scan_id, (None, None))[0],
            "y": reference_xy_by_scan_id.get(scan_id, (None, None))[1],
            "label_available": scan_id in reference_xy_by_scan_id,
        }
        for scan_id in scan_ids
    ]
).set_index("scan_id")

updated_fingerprint_frame = pd.DataFrame(
    updated_rssi_dbm.numpy(),
    index=pd.Index(scan_ids, name="scan_id"),
    columns=pd.Index(bssid_vocabulary, name="bssid"),
)
assert tuple(updated_fingerprint_frame.index) == scan_ids
assert tuple(updated_fingerprint_frame.columns) == bssid_vocabulary

aligned_reference_table = coordinate_frame.join(
    updated_fingerprint_frame,
    how="left",
    validate="one_to_one",
)
assert tuple(aligned_reference_table.index) == scan_ids
localization_training_table = aligned_reference_table.loc[
    aligned_reference_table["label_available"]
].copy()
assert len(localization_training_table) == len(reference_position_ids)

print(f"All updated reference rows: {aligned_reference_table.shape}")
print(f"Labeled localization rows: {localization_training_table.shape}")
display(aligned_reference_table.head())


All updated reference rows: (2078, 1044)
Labeled localization rows: (2078, 1044)


,x,y,label_available,000B819A90BF,000B819BBC05,000B819EEBA5,000B819FDD48,000B81A08E12,000B81A19DC9,000B81A2359A,...,60F18AD659AB,60F18AD659AF,60F18AD6C66B,60F18AD6D0CB,60F18AD6D0CF,8235C1BA6A69,BA99666F337A,D6F15F11F88A,DA9C679B9286,EC8914872F3F
scan_id,,,,,,,,,,,,,,,,,,,,,
scan:dc7f29deee0a3ef0a656e7bf575d36157e772603f07f1737cc362d8ae532d3fe,458.651806,392.645723,True,-119.903015,-119.696701,-118.642502,-119.877296,-119.743279,-88.500000,-116.960007,...,-120.520912,-119.949844,-118.746399,-116.435974,-119.722107,-119.876305,-119.637459,-119.317665,-119.605400,-119.393616
scan:3d6e7516fb396a8f466059332dea349c7129ca3e2356a61a8355e6a8bc58373c,458.731986,396.680500,True,-119.900055,-119.660408,-118.623268,-119.867966,-119.742149,-112.292221,-116.976707,...,-120.516930,-119.948151,-118.751549,-116.452583,-119.736504,-119.876297,-119.659218,-119.327171,-119.624557,-119.397072
scan:62b353a32f5730f8a9720917fe5e179fdddb361e83851bb4a0cbc54556bff142,460.125183,380.536226,True,-119.914742,-119.713806,-118.727356,-119.912109,-119.750870,-112.401802,-116.833031,...,-120.514053,-119.954529,-118.800674,-116.517784,-119.699471,-119.877777,-119.620514,-119.341354,-119.630150,-119.402634
scan:9e6a20e0ecae343970251cdf768c0e5f8bd30ef4596f3d9833edf2149326abc5,460.508937,384.570088,True,-119.910370,-119.696136,-118.717499,-119.908768,-119.746498,-112.280457,-116.802628,...,-120.511391,-119.955467,-118.808937,-116.527588,-119.701881,-119.880890,-119.641876,-119.352074,-119.640030,-119.405106
scan:93f6c79a1172aa19f41501ee563732025b8ec4a74726b7d51e4c0b2f44c9157b,460.589117,388.604864,True,-119.907768,-119.685089,-118.699936,-119.903946,-119.743591,-112.256935,-116.829018,...,-120.509659,-119.954445,-118.809830,-116.529533,-119.709442,-119.881683,-119.659348,-119.356667,-119.648720,-119.408028


In [11]:
aligned_reference_table

,x,y,label_available,000B819A90BF,000B819BBC05,000B819EEBA5,000B819FDD48,000B81A08E12,000B81A19DC9,000B81A2359A,...,60F18AD659AB,60F18AD659AF,60F18AD6C66B,60F18AD6D0CB,60F18AD6D0CF,8235C1BA6A69,BA99666F337A,D6F15F11F88A,DA9C679B9286,EC8914872F3F
scan_id,,,,,,,,,,,,,,,,,,,,,
scan:dc7f29deee0a3ef0a656e7bf575d36157e772603f07f1737cc362d8ae532d3fe,458.651806,392.645723,True,-119.903015,-119.696701,-118.642502,-119.877296,-119.743279,-88.500000,-116.960007,...,-120.520912,-119.949844,-118.746399,-116.435974,-119.722107,-119.876305,-119.637459,-119.317665,-119.605400,-119.393616
scan:3d6e7516fb396a8f466059332dea349c7129ca3e2356a61a8355e6a8bc58373c,458.731986,396.680500,True,-119.900055,-119.660408,-118.623268,-119.867966,-119.742149,-112.292221,-116.976707,...,-120.516930,-119.948151,-118.751549,-116.452583,-119.736504,-119.876297,-119.659218,-119.327171,-119.624557,-119.397072
scan:62b353a32f5730f8a9720917fe5e179fdddb361e83851bb4a0cbc54556bff142,460.125183,380.536226,True,-119.914742,-119.713806,-118.727356,-119.912109,-119.750870,-112.401802,-116.833031,...,-120.514053,-119.954529,-118.800674,-116.517784,-119.699471,-119.877777,-119.620514,-119.341354,-119.630150,-119.402634
scan:9e6a20e0ecae343970251cdf768c0e5f8bd30ef4596f3d9833edf2149326abc5,460.508937,384.570088,True,-119.910370,-119.696136,-118.717499,-119.908768,-119.746498,-112.280457,-116.802628,...,-120.511391,-119.955467,-118.808937,-116.527588,-119.701881,-119.880890,-119.641876,-119.352074,-119.640030,-119.405106
scan:93f6c79a1172aa19f41501ee563732025b8ec4a74726b7d51e4c0b2f44c9157b,460.589117,388.604864,True,-119.907768,-119.685089,-118.699936,-119.903946,-119.743591,-112.256935,-116.829018,...,-120.509659,-119.954445,-118.809830,-116.529533,-119.709442,-119.881683,-119.659348,-119.356667,-119.648720,-119.408028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
scan:8b7ed5749cb9b6911af3d1371504a3f48779e6efe9a81a3960173d3783d58eb4,425.355945,477.011267,True,-119.860649,-119.329277,-117.188492,-119.446854,-119.949730,-116.843575,-119.599518,...,-119.515549,-119.919159,-116.236351,-112.669220,-118.072433,-119.873886,-117.589050,-117.189789,-117.059761,-118.790756
scan:0579578d658cf4e1443d5e4cf9de8e711e4f7ac028a2fd8984460e187cdbac6b,429.338330,476.361780,True,-119.855797,-119.214798,-116.959053,-119.465904,-119.925987,-116.457428,-119.382172,...,-119.669106,-119.922058,-116.556168,-113.240150,-118.213387,-119.887932,-117.818634,-117.433533,-117.409996,-118.824402
scan:e6b29fb988eb29c96d6c432a9cd0234fce78b5972ad556c32a3756331ff25a1d,433.400186,477.310843,True,-119.855133,-119.092255,-116.765671,-119.469513,-119.906548,-116.132164,-119.180504,...,-119.809929,-119.923409,-116.880684,-113.838150,-118.323326,-119.905273,-118.059090,-117.663742,-117.746742,-118.867012


In [12]:
from unknot_data.base import GeoLocationData
from unknot_data.vector import WifiBssidVector, WifiRssiVector
from unknot_data.wifi_networks import WifiNetwork
from unknot_data.rssi_index import RssiIndex, RssiIntegerIndex
from unknot_data.data_viz import AxisGrid
from unknot_data.utils import load_scan_from_txt

In [18]:
bssids, poses = load_scan_from_txt(
    source_path="input/GF_fromGF_12/fingerprint0.txt",
    geo_reference = GeoLocationData(longitude=-74.6520576, latitude=40.3637847, altitude=0.0)
)

wifi_network = WifiNetwork(
    bssids=sorted({b for bssid in bssids[:-100] for b in bssid.bssid})
)

In [19]:
bssids_val, poses_val = load_scan_from_txt(
    source_path="input/GF_fromGF_12/fingerprint1.txt",
    geo_reference = GeoLocationData(longitude=-74.6520576, latitude=40.3637847, altitude=0.0)
)
rssis_val = wifi_network.bssid2rssi(bssids_val)

In [20]:
import numpy as np

bssids_train = bssids[:-100]
poses_train = poses[:-100]
rssis_train = wifi_network.bssid2rssi(bssids_train)

rssi_index = RssiIndex(
    positions=np.stack([pose.arr() for pose in poses_train]),
    rssis = np.stack([rssi.arr() for rssi in rssis_train]),
    dim=len(wifi_network.bssids)
)

In [21]:
bssids_test = bssids[-100:]
poses_test = poses[-100:]
rssis_test = wifi_network.bssid2rssi(bssids_test)

In [22]:
preds = np.stack([rssi_index.predict(rssi.arr(), k=10) for rssi in rssis_test])
targs = np.stack([pose.arr() for pose in poses_test])

error_test = np.linalg.norm(preds-targs, axis=-1).mean()
print(error_test)

35.37371846167573


In [23]:
preds = np.stack([rssi_index.predict(rssi.arr(), k=10) for rssi in rssis_val])
targs = np.stack([pose.arr() for pose in poses_val])

error_val = np.linalg.norm(preds-targs, axis=-1).mean()
print(error_val)

20.02935732337639


In [43]:
from pathlib import Path
import numpy as np
from gufu_data import _parse_db_record
from unknot_data.base import GeoLocationData
from unknot_data.timestamped_pose import TimestampedGeoPose, TimestampedPose
from unknot_data.vector import WifiBssidVector, WifiRssiVector
from unknot_data.wifi_networks import WifiNetwork
from unknot_data.rssi_index import RssiIndex, RssiIntegerIndex
from unknot_data.utils import normalize_rssi
from unknot_data.utils import load_scan_from_txt
from unknot_data.data_viz import AxisGrid

zero_threshold = 20
positions = dict(zip(
    aligned_reference_table.index.tolist(),
    aligned_reference_table[['x', 'y']].values.tolist()
))

# rssis = {scan.scan_id: scan.rss_dbm for scan in updated_reference.scans}

rssis = dict(zip(
    round2_fingerprint_frame.index.tolist(),
    round2_fingerprint_frame.values.tolist()
))


vocab = np.array(bssid_vocabulary)
geo_reference = GeoLocationData(longitude=-74.6520576, latitude=40.3637847, altitude=0.0)

bssids2, poses2 = [], []

for i, (scan_id, xy) in enumerate(positions.items()):
    scan = np.array(rssis[scan_id])

    bssid = WifiBssidVector(
        timestamp=i,
        V=[normalize_rssi(x) for x in scan[scan>-120 + zero_threshold].tolist()],
        bssid=[x for x in vocab[scan>-120 + zero_threshold].tolist()],
    )
    pose = TimestampedPose(
        timestamp=i,
        x=xy[0],
        y=xy[1],
        z=0.0,
        qw=1.0,
        qx=0.0,
        qy=0.0,
        qz=0.0,
    ).to_timestamped_geo_pose(geo_reference)

    bssids2.append(bssid)
    poses2.append(pose)


In [44]:
wifi_network2 = WifiNetwork(
    bssids=list(bssid_vocabulary)
)

In [45]:
import numpy as np

bssids_train2 = bssids2[:-100]
poses_train2 = poses2[:-100]
rssis_train2 = wifi_network2.bssid2rssi(bssids_train2)

rssi_index2 = RssiIndex(
    positions=np.stack([pose.arr() for pose in poses_train2]),
    rssis = np.stack([rssi.arr() for rssi in rssis_train2]),
    dim=len(wifi_network2.bssids)
)

In [46]:
bssids_val2, poses_val2 = load_scan_from_txt(
    source_path="input/GF_fromGF_12/fingerprint1.txt",
    geo_reference = GeoLocationData(longitude=-74.6520576, latitude=40.3637847, altitude=0.0)
)
rssis_val2 = wifi_network2.bssid2rssi(bssids_val2)

In [33]:
rssi_index.knn_query(np.array(rssis_val[0].V), 10, return_distances=True)

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(10, 992), dtype=float32),
 array([[243.46004, 493.41736],
        [243.44553, 476.88565],
        [233.52927, 473.1009 ],
        [240.26048, 472.05862],
        [230.55939, 489.1665 ],
        [239.00809, 492.4351 ],
        [243.16847, 484.96567],
        [239.10303, 495.47348],
        [219.48152, 467.4096 ],
        [226.21854, 473.01654]], dtype=float32),
 array([[0.05959731, 0.07002544, 0.0748055 , 0.08142608, 0.08368111,
         0.08393282, 0.08436275, 0.08703947, 0.08761561, 0.08904612]],
       dtype=float32))

In [34]:
rssi_index2.knn_query(np.array(rssis_val2[0].V), 10, return_distances=True)

(array([[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]], shape=(10, 1041), dtype=float32),
 array([[243.46004, 493.41736],
        [243.44553, 476.88565],
        [282.01987, 471.92697],
        [279.88852, 480.09793],
        [268.7402 , 494.84448],
        [229.72173, 493.4903 ],
        [269.50818, 471.39966],
        [233.52927, 473.1009 ],
        [239.00809, 492.4351 ],
        [297.99042, 495.68137]], dtype=float32),
 array([[0.2798183 , 0.28104407, 0.29472125, 0.29809868, 0.29848588,
         0.29946095, 0.3001626 , 0.30053413, 0.30086303, 0.30155516]],
       dtype=float32))

In [48]:
preds = np.stack([rssi_index2.predict(rssi.arr(), k=10) for rssi in rssis_val2])
targs = np.stack([pose.arr() for pose in poses_val2])

error_val = np.linalg.norm(preds-targs, axis=-1).mean()
print(error_val)

21.943026831852492


In [35]:
preds = np.stack([rssi_index2.predict(rssi.arr(), k=30) for rssi in rssis_val2])
targs = np.stack([pose.arr() for pose in poses_val2])

error_val = np.linalg.norm(preds-targs, axis=-1).mean()
print(error_val)

22.145093998308752


In [49]:
preds = np.stack([rssi_index.predict(rssi.arr(), k=10) for rssi in rssis_val])
targs = np.stack([pose.arr() for pose in poses_val])

error_val = np.linalg.norm(preds-targs, axis=-1).mean()
print(error_val)

20.02935732337639


## Step 11 — Compare fingerprint update rounds

For streaming snapshots, round 1 is the post-transition AE fingerprint output already loaded above. Round 2 is a separate candidate produced by applying the final GNN once to the rebuilt reference+update graph, training another location/feature updater, and running Eq. (10) again. The candidate is saved for comparison and is not installed as the pipeline's current reference state. This section verifies both artifacts and compares aligned scan/AP cells.


['scan:dc7f29deee0a3ef0a656e7bf575d36157e772603f07f1737cc362d8ae532d3fe',
 'scan:3d6e7516fb396a8f466059332dea349c7129ca3e2356a61a8355e6a8bc58373c',
 'scan:62b353a32f5730f8a9720917fe5e179fdddb361e83851bb4a0cbc54556bff142',
 'scan:9e6a20e0ecae343970251cdf768c0e5f8bd30ef4596f3d9833edf2149326abc5',
 'scan:93f6c79a1172aa19f41501ee563732025b8ec4a74726b7d51e4c0b2f44c9157b',
 'scan:b842e5dabb08aeaeed3b1e1daf9879e3ed26177d435a7558a84d7d89634c705e',
 'scan:affa23d46be5a64ebb3bdfb4f43df81c793ad04536c0d0b595909836f6c76eb6',
 'scan:25f922e32b936ce294de1f4ea4f0ef371e0dcf7f5da410862cc76844912757c0',
 'scan:cdd49d3e2fff8a4fe87c0cfd33038f1898e05c2841f0977a3d87bd890ce32610',
 'scan:23608b95a3fe59e86a12ffe5b89a81c980795e8f11eb1f8cb7d3dcfa05413785',
 'scan:d8edc90e68113142e644dba9818108f0c3b58cff98145d047d03b754dac08504',
 'scan:25f9d954d52c1c542e982614a4d1523f497c9d9107670660bdd2a231ff4eb707',
 'scan:762df271b8228707ea622e16bf4d9950528bbe8552515f509105d28f0dd44c8e',
 'scan:c358361727f1059b3948235f8d92765

[[-119.78361511230469,
  -119.53709411621094,
  -118.27379608154297,
  -119.57891082763672,
  -119.65324401855469,
  -88.5,
  -114.76283264160156,
  -108.3741683959961,
  -119.83231353759766,
  -119.41572570800781,
  -119.95227813720703,
  -119.9036865234375,
  -119.78995513916016,
  -119.82070922851562,
  -119.4095458984375,
  -119.62665557861328,
  -119.77154541015625,
  -119.77342987060547,
  -118.73291778564453,
  -119.67127990722656,
  -120.02888488769531,
  -119.77871704101562,
  -113.37594604492188,
  -114.647216796875,
  -119.51370239257812,
  -119.47640991210938,
  -119.9356689453125,
  -119.849609375,
  -119.80794525146484,
  -119.5489501953125,
  -119.85883331298828,
  -119.84678649902344,
  -117.28744506835938,
  -117.84616088867188,
  -119.81743621826172,
  -119.76251220703125,
  -119.13775634765625,
  -119.83199310302734,
  -119.88336944580078,
  -119.39268493652344,
  -120.16252136230469,
  -119.91171264648438,
  -119.51366424560547,
  -119.74088287353516,
  -119.9264907

In [37]:
if snapshot_format != "lp_streaming":
    print("Round-by-round comparison is available for lp_streaming snapshots only.")
    round1_rssi_dbm = None
    round2_rssi_dbm = None
    round_comparison = None
else:
    from streaming_fingerprint_update import (
        StreamingAEUpdateManifest,
        StreamingPostTransitionAEManifest,
    )

    round1_directory = SNAPSHOT_DIRECTORY / "post_transition_ae"
    round2_directory = SNAPSHOT_DIRECTORY / "post_transition_fingerprint_update_round2"
    round2_record = json.loads(
        (round2_directory / "round2_manifest.json").read_text(encoding="utf-8")
    )
    assert round2_record["format"] == "gufu_post_transition_fingerprint_update_round2_v1"
    assert round2_record["role"] == "comparison_candidate"
    assert tuple(round2_record["reference_scan_ids"]) == scan_ids
    assert tuple(round2_record["bssid_vocabulary"]) == bssid_vocabulary
    assert round2_record["round1_reference_prepared_sha256"] == canonical_json_digest(updated_reference)

    round1_manifest_path = round1_directory / "manifest.json"
    round1_manifest = StreamingPostTransitionAEManifest.model_validate_json(
        round1_manifest_path.read_text(encoding="utf-8")
    )
    round1_marker = json.loads(
        (round1_directory / "COMPLETED.json").read_text(encoding="utf-8")
    )
    assert round1_marker["identity"] == round1_manifest.post_update_id
    assert round1_marker["manifest_sha256"] == sha256_file(round1_manifest_path)
    round1_array_record = next(
        item for item in round1_manifest.arrays if item.name == "updated_rssi_dbm.npy"
    )
    round1_array_path = round1_directory / round1_array_record.name
    assert sha256_file(round1_array_path) == round1_array_record.sha256
    round1_rssi_dbm = np.load(round1_array_path, mmap_mode="r", allow_pickle=False)

    round2_ae_directory = round2_directory / round2_record["ae_update_directory"]
    round2_manifest_path = round2_ae_directory / "manifest.json"
    round2_manifest = StreamingAEUpdateManifest.model_validate_json(
        round2_manifest_path.read_text(encoding="utf-8")
    )
    round2_marker = json.loads(
        (round2_ae_directory / "COMPLETED.json").read_text(encoding="utf-8")
    )
    assert round2_marker["identity"] == round2_manifest.update_id
    assert round2_marker["manifest_sha256"] == sha256_file(round2_manifest_path)
    assert round2_manifest.update_id == round2_record["ae_update_id"]
    assert round2_manifest.updater_id == round2_record["updater_id"]
    assert round2_manifest.online_graph_id == round2_record["query_graph_id"]
    round2_array_record = next(
        item for item in round2_manifest.arrays if item.name == "updated_rssi_dbm.npy"
    )
    round2_array_path = round2_ae_directory / round2_array_record.name
    assert sha256_file(round2_array_path) == round2_array_record.sha256
    assert sha256_file(round2_array_path) == round2_record["updated_rssi_dbm_sha256"]
    round2_rssi_dbm = np.load(round2_array_path, mmap_mode="r", allow_pickle=False)

    expected_shape = (len(scan_ids), len(bssid_vocabulary))
    assert round1_rssi_dbm.shape == expected_shape
    assert round2_rssi_dbm.shape == expected_shape
    assert tuple(round1_manifest.reference_scan_ids) == scan_ids
    assert tuple(round1_manifest.bssid_vocabulary) == bssid_vocabulary
    assert tuple(round2_manifest.reference_scan_ids) == scan_ids
    assert tuple(round2_manifest.bssid_vocabulary) == bssid_vocabulary
    assert np.allclose(
        round1_rssi_dbm,
        updated_rssi_dbm.numpy(),
        rtol=0.0,
        atol=1e-5,
    ), "round 1 artifact differs from the prepared reference state"

    round1_fingerprint_frame = pd.DataFrame(
        np.asarray(round1_rssi_dbm),
        index=pd.Index(scan_ids, name="scan_id"),
        columns=pd.Index(bssid_vocabulary, name="bssid"),
    )
    round2_fingerprint_frame = pd.DataFrame(
        np.asarray(round2_rssi_dbm),
        index=pd.Index(scan_ids, name="scan_id"),
        columns=pd.Index(bssid_vocabulary, name="bssid"),
    )
    round_delta_frame = round2_fingerprint_frame - round1_fingerprint_frame
    measured = observation_mask.numpy()
    inferred = ~measured
    assert np.allclose(
        np.asarray(round2_rssi_dbm)[measured],
        np.asarray(round1_rssi_dbm)[measured],
        rtol=0.0,
        atol=1e-5,
    ), "round 2 changed physically measured RSSI values"

    round_comparison = pd.DataFrame(
        {
            "scan_id": scan_ids,
            "mean_abs_delta_all_cells_dbm": np.mean(
                np.abs(np.asarray(round_delta_frame)), axis=1
            ),
            "mean_abs_delta_inferred_cells_dbm": [
                float(np.abs(np.asarray(round_delta_frame)[row][inferred[row]]).mean())
                if inferred[row].any()
                else 0.0
                for row in range(len(scan_ids))
            ],
            "max_abs_delta_inferred_cells_dbm": [
                float(np.abs(np.asarray(round_delta_frame)[row][inferred[row]]).max())
                if inferred[row].any()
                else 0.0
                for row in range(len(scan_ids))
            ],
        }
    ).set_index("scan_id")
    print(
        f"Round 1: {round1_manifest.post_update_id}; "
        f"round 2 candidate: {round2_manifest.update_id}"
    )
    print(
        "Across all cells: mean |round2 - round1| = "
        f"{np.abs(np.asarray(round_delta_frame)).mean():.4f} dBm; "
        "physically measured cells are unchanged."
    )
    display(round_comparison.sort_values("mean_abs_delta_inferred_cells_dbm", ascending=False).head(20))

    comparison_scan_id = scan_ids[example_row]
    comparison_fingerprint = pd.DataFrame(
        {
            "round1_rssi_dbm": round1_fingerprint_frame.loc[comparison_scan_id],
            "round2_rssi_dbm": round2_fingerprint_frame.loc[comparison_scan_id],
            "round2_minus_round1_dbm": round_delta_frame.loc[comparison_scan_id],
            "measured_in_original_scan": observation_mask[example_row].numpy(),
        },
        index=pd.Index(bssid_vocabulary, name="bssid"),
    ).sort_values("round2_minus_round1_dbm", key=lambda values: values.abs(), ascending=False)
    print(f"Largest cell changes for scan {comparison_scan_id}:")
    display(comparison_fingerprint.head(25))


Round 1: streampostae:663af184a5c2e3448a6e532727d744605a3ade663132949517533ed50602692b; round 2 candidate: streamae:0e525cdabcddd68a4deb723845f6fdb815d40fbdf2d75671ac5efd2a8339759d
Across all cells: mean |round2 - round1| = 0.4562 dBm; physically measured cells are unchanged.


,mean_abs_delta_all_cells_dbm,mean_abs_delta_inferred_cells_dbm,max_abs_delta_inferred_cells_dbm
scan_id,,,
scan:8eb035bced70213b816cd8105fc657fc75e309fd22c6038c179c3f1ad901818a,0.872460,0.892172,8.225380
scan:914290e16a0b6ee8eeea5b20cc86b199cb8a6ac08d0f2fc09c7fa63ff6791dae,0.854126,0.891821,9.579048
scan:953d722ede810e189a9ebece4475d53c1f5e9faea537c0d7f3342336666c6701,0.834769,0.868125,8.765594
scan:432c4a29db244e4a03ced2560d415e003175360c4e92ebcd4ddc8ac31223d266,0.847431,0.862341,10.695511
scan:ea62a0f4f5aa4c7df19b168702a89656c276f809f3de8b669faa737dc4abbb7e,0.833430,0.857313,8.151237
scan:b9093304cfc96f178baf8c7216f3f4efbce08b325c8d3dd875967ee2b038647a,0.831710,0.851337,8.144020
scan:2e39f9c324f30c634b26bb5548032969aa8891595dc0b5bd00c2d4c7da95cb99,0.831814,0.849772,10.684975
scan:6417d477aebee9660abc80f0b85186555e329834fbe9f79f5754e42688a40225,0.823735,0.849018,8.021606
scan:492a55e26d2b052ae26b9e77a556030f5d50528a46daaef387ddc012e807b028,0.795499,0.841580,8.908150


Largest cell changes for scan scan:dc7f29deee0a3ef0a656e7bf575d36157e772603f07f1737cc362d8ae532d3fe:


,round1_rssi_dbm,round2_rssi_dbm,round2_minus_round1_dbm,measured_in_original_scan
bssid,,,,
6CAAB3127669,-107.820526,-101.901947,5.918579,False
34A84EC7B164,-103.676361,-97.842896,5.833466,False
4CEDFBA8FDD8,-110.287514,-104.616150,5.671364,False
BC26C795D9E4,-106.698601,-101.105400,5.593201,False
6CAAB3527668,-108.732643,-103.296936,5.435707,False
0C8063236353,-109.124702,-103.751213,5.373489,False
44D1FA2020CC,-103.397667,-98.066925,5.330742,False
14918261EA83,-108.867027,-103.587456,5.279572,False
6CAAB3127668,-110.228477,-104.952820,5.275658,False
